# Token Model — Byte-Level BPE + RoPE

Two architectural upgrades over the baseline:
* **Byte-level BPE tokenizer** (`tokenizers.ByteLevelBPETokenizer`) — sub-word units shared across identifiers (`self.foo` and `self.bar` reuse `self.`), better generalisation, smaller vocabulary needed for the same coverage.
* **Rotary Position Embeddings (RoPE)** — applied to Q and K inside attention; generalises to longer contexts and plays nicely with `F.scaled_dot_product_attention` (Flash-Attn / mem-efficient kernels).

Everything else (warmup+cosine LR, label smoothing, mixed precision, early stopping) is unchanged.

In [10]:
%tb
import os, math, random, glob
from pathlib import Path
from dataclasses import dataclass
from typing import List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup


import warnings
warnings.filterwarnings("ignore")

from modules.plotting import MetricLog, plot_metrics
from modules.early_stopping import EarlyStopping
from modules.hand_testing import hand_test_repl
from modules.best_model_saver import BestModelSaver
from modules.datasets.loading import *
from modules.datasets.token_dataset import *
from modules.tokenizers.BPE_tokenizer import *
from modules.models.T_rope_model import *

WORKDIR = r'C:\Users\Roman\Documents\Projects\code_autocomplete'
print(f"WORKDIR: {WORKDIR}")
TOKEN_MODEL_NAME = 'token_model_bpe_rope'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")

NameError: name 'TokenDataset' is not defined

WORKDIR: C:\Users\Roman\Documents\Projects\code_autocomplete
[Device] cuda


## Training loop

In [11]:
def _clip_norm(model, max_norm=1.0):
    return nn.utils.clip_grad_norm_(model.parameters(), max_norm).item()


def train_token_model(model, train_dl, val_dl, epochs, lr, device,
                      saver, log, plot_dir,
                      label_smoothing=0.1, warmup_frac=0.05,
                      patience=3, use_amp=True):
    tqdm.write(f"[Token] DataLoader — {len(train_dl)} train batches, {len(val_dl)} val batches")
    opt          = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    total_steps  = len(train_dl) * epochs
    warmup_steps = max(1, int(total_steps * warmup_frac))
    sched        = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)
    crit         = nn.CrossEntropyLoss(ignore_index=SPECIAL["<PAD>"], label_smoothing=label_smoothing)
    amp_enabled  = use_amp and device.type == "cuda"
    scaler       = GradScaler("cuda", enabled=amp_enabled)
    stopper      = EarlyStopping(patience=patience)

    for ep in range(1, epochs + 1):
        print(f"Epoch {ep}")
        model.train()
        t_loss = t_acc = t_steps = 0; gn = 0.0
        for x, y in tqdm(train_dl, desc=f"[Token] Epoch {ep}/{epochs} train",
                         leave=False, unit="batch"):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast("cuda", dtype=torch.float16, enabled=amp_enabled):
                logits = model(x)
                loss   = crit(logits.view(-1, logits.size(-1)), y.view(-1))
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            gn = _clip_norm(model)
            scaler.step(opt); scaler.update(); sched.step()
            with torch.no_grad():
                preds = logits.argmax(-1)
                mask  = (y != SPECIAL["<PAD>"])
                t_acc += (preds[mask] == y[mask]).float().mean().item()
            t_loss += loss.item(); t_steps += 1
        tl, ta = t_loss / t_steps, t_acc / t_steps

        model.eval()
        v_loss = v_steps = 0
        with torch.no_grad():
            for x, y in tqdm(val_dl, desc=f"[Token] Epoch {ep}/{epochs} val  ",
                             leave=False, unit="batch"):
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                with autocast("cuda", dtype=torch.float16, enabled=amp_enabled):
                    logits = model(x)
                    loss   = crit(logits.view(-1, logits.size(-1)), y.view(-1))
                v_loss += loss.item(); v_steps += 1
        vl = v_loss / v_steps if v_steps else tl

        log.append(train_loss=tl, val_loss=vl,
                   train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
                   lr=opt.param_groups[0]["lr"], token_acc=ta, grad_norm=gn)
        tqdm.write(f"[Token ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
                   f"  ppl={math.exp(min(vl,20)):.1f}  acc={ta:.3f}"
                   f"  lr={opt.param_groups[0]['lr']:.2e}")
        saver.save(model, vl, ep)
        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(log, f"{TOKEN_MODEL_NAME.replace('_', ' ')} — Epoch {ep}",
                         f"{plot_dir}/{TOKEN_MODEL_NAME}_ep{ep:02d}.png")
        if stopper(vl):
            tqdm.write(f"[Early stop] val_loss did not improve for {stopper.patience} epochs — stopping.")
            break
    plot_metrics(log, f"{TOKEN_MODEL_NAME.replace('_', ' ')} — Final",
                 f"{plot_dir}/{TOKEN_MODEL_NAME}_final.png")

## Main

In [12]:
class Arguments():
    def __init__(self,
                 data_dir=f"{WORKDIR}/Clean_Dataset",
                 ckpt_dir=f"{WORKDIR}/checkpoints/{TOKEN_MODEL_NAME}",
                 plot_dir=f"{WORKDIR}/plots/{TOKEN_MODEL_NAME}",
                 tokenizer=f"{WORKDIR}/tokenizer_bpe.json",
                 epochs=5, batch=32, lr=5e-4, ctx=128,
                 d_model=256, n_layers=4, n_heads=8,
                 vocab_size=16000, max_files=0, val_split=0.1, seed=42,
                 label_smoothing=0.1, warmup_frac=0.05, patience=3, use_amp=True,
                 skip_token=False, test=False):
        for k, v in locals().items():
            if k != "self": setattr(self, k, v)


def main():
    # args = Arguments()
    args = Arguments(max_files=5, epochs=2)
    # args = Arguments(test=True)

    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(args.seed)
    os.makedirs(args.ckpt_dir, exist_ok=True)
    os.makedirs(args.plot_dir, exist_ok=True)

    if os.path.exists(args.tokenizer):
        print(f"[Tokenizer] loading {args.tokenizer}")
        tokenizer = BPECodeTokenizer.load(args.tokenizer)
    else:
        print("[Tokenizer] training byte-level BPE from data …")
        texts = load_files(args.data_dir, args.max_files)
        if not texts: print("[ERROR] no data files found."); return
        tokenizer = BPECodeTokenizer(vocab_size=args.vocab_size)
        tokenizer.build(texts)
        tokenizer.save(args.tokenizer)
        print(f"[Tokenizer] saved to {args.tokenizer}")

    cfg = ModelCfg(
        vocab=tokenizer.vocab, d_model=args.d_model,
        n_heads=args.n_heads, n_layers=args.n_layers,
        d_ff=args.d_model * 4, max_len=args.ctx + 32,
    )
    torch.serialization.add_safe_globals([ModelCfg])

    if args.test:
        tm = TokenModel(cfg).to(device)
        tok_paths = sorted(glob.glob(str(Path(args.ckpt_dir) / f"{TOKEN_MODEL_NAME}_*.pt")))
        if tok_paths:
            ck = torch.load(tok_paths[0], map_location=device, weights_only=False)
            tm.load_state_dict(ck["model_state"])
            print(f"[Loaded] token model from {tok_paths[0]}")
        hand_test_repl(tm, None, tokenizer, None, device); return

    print("[Loading] Started loading")
    texts = load_files(args.data_dir, args.max_files)
    if not texts: print("[ERROR] no data files found."); return
    print("[Loading] Ended loading")

    random.shuffle(texts)
    split = max(1, int(len(texts) * (1 - args.val_split)))
    tr_txt, va_txt = texts[:split], texts[split:]

    if not args.skip_token:
        print("  Preparing TOKEN model (BPE + RoPE)")
        all_ids_tr = []
        for t in tqdm(tr_txt, desc="[Encode train]", unit="file"):
            all_ids_tr.extend(tokenizer.encode(t))
        all_ids_va = []
        for t in tqdm(va_txt, desc="[Encode val]  ", unit="file"):
            all_ids_va.extend(tokenizer.encode(t))
        print(f"  Train tokens: {len(all_ids_tr):,}  |  Val tokens: {len(all_ids_va):,}")

        tr_ds = TokenDataset(all_ids_tr, args.ctx)
        va_ds = TokenDataset(all_ids_va, args.ctx)
        tr_dl = DataLoader(tr_ds, args.batch, shuffle=True,  num_workers=0, pin_memory=True)
        va_dl = DataLoader(va_ds, args.batch, shuffle=False, num_workers=0, pin_memory=True)

        tok_model = TokenModel(cfg).to(device)
        n_params  = sum(p.numel() for p in tok_model.parameters() if p.requires_grad)
        print(f"[Token Model] {n_params/1e6:.2f}M parameters (BPE+RoPE)")

        tok_saver = BestModelSaver(args.ckpt_dir, TOKEN_MODEL_NAME)
        tok_log   = MetricLog()
        print("  Training TOKEN model")
        train_token_model(
            model=tok_model, train_dl=tr_dl, val_dl=va_dl,
            epochs=args.epochs, lr=args.lr, device=device,
            saver=tok_saver, log=tok_log, plot_dir=args.plot_dir,
            label_smoothing=args.label_smoothing, warmup_frac=args.warmup_frac,
            patience=args.patience, use_amp=args.use_amp,
        )
        hand_test_repl(tok_model, None, tokenizer, None, device)


main()

[Tokenizer] loading C:\Users\Roman\Documents\Projects\code_autocomplete/tokenizer_bpe.json
[Loading] Started loading
[Data] loaded 5 files from C:\Users\Roman\Documents\Projects\code_autocomplete/Clean_Dataset
[Loading] Ended loading
  Preparing TOKEN model (BPE + RoPE)


[Encode val]  : 100%|██████████| 1/1 [00:00<00:00, 171.22file/s]

  Train tokens: 4,034  |  Val tokens: 2,846


[Token Model] 5.33M parameters (BPE+RoPE)
  Training TOKEN model
[Token] DataLoader — 123 train batches, 85 val batches
Epoch 1


[Token ep   1] train_loss=4.9416  val_loss=7.3645  ppl=1578.9  acc=0.342  lr=2.70e-04
[Saver] saved ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\token_model_bpe_rope\token_model_bpe_rope_ep001_loss7.3645.pt  (val_loss=7.3645)
[Plot] saved → C:\Users\Roman\Documents\Projects\code_autocomplete/plots/token_model_bpe_rope/token_model_bpe_rope_ep01.png
Epoch 2


[Token ep   2] train_loss=2.1739  val_loss=7.6488  ppl=2098.2  acc=0.812  lr=0.00e+00
[Saver] saved ckpt: C:\Users\Roman\Documents\Projects\code_autocomplete\checkpoints\token_model_bpe_rope\token_model_bpe_rope_ep002_loss7.6488.pt  (val_loss=7.6488)
[Plot] saved → C:\Users\Roman\Documents\Projects\code_autocomplete/plots/token_model_bpe_rope/token_model_bpe_rope_ep02.png
[Plot] saved → C:\Users\Roman\Documents\Projects\code_autocomplete/plots/token_model_bpe_rope/token_model_bpe_rope_final.png
